# Schelling's Segregation Model 
# with Matplotlib Inline Visualisation for Jupyter Notebooks

When using Matplotlib for interactive visualisations in this form, it is important to bear in mind that almost all errors generated by functions called by the interactive interface are masked / hidden, i.e. they will interupt the execution of the calling function but no error message will be visible. This makes debugging extremely difficult! It is thus crucial to find a way to avoid this during development. One possibility to achieve this is to "layer" the development, i.e. to explicitly call the step() and draw() function from the command line (Jupyter cell) first without the interactive interface (This will leave error messages visible). Only in the final step, when the update functions work, should the interactive interface be wrapped around everything.  

The alternative is to use TKAgg, in which case matplotlib will run in a separate window.
This can be a bit snappier.
Note, however, that the current way the animation is implemented via a timer does not work with TkAgg. Manual stepping works fine, but auto-stepping doesn't. Messing around with timed functions in Tkagg is a bit tricky, because the Tk main loop takes over and GUI functions can only be triggered from the main thread (see https://www.physics.utoronto.ca/~phy326/python/Live_Plot.py). The alternative, to use Matplotlib animation, does not work either, because it cannot be interrupted. 

## Set up matplotlib interface

Either choose ipympl or TkAgg as the Matplotlib backend but onbly call one, not both!

ipympl keeps the visualisation within the notebook and should work universally (including in browser-based Jupyter servers), however, it does not support auto-running. Single-step execution should work fine. TkAgg supports auto-running but will normally not working in web-based Jupyter severs. 

In [1]:
%matplotlib ipympl

**Do not !!! execute the next line first if you want to use ipympl**

**If you want to use TkAgg you need to uncomment the code first (but in this case, do not execute the line "%matplotlib ipympl")**

In [1]:
import matplotlib
matplotlib.use('TkAgg')

## General Imports

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Button, Slider
import matplotlib.cm as cm

import random as RD
import scipy as SP

import time, threading

## The actual simulation code

In [6]:
def simulation(density, threshold):
    global config, fig, ax, bnext, bstart, bstop, binit, empty, unhappiness, avg_similarity
    # apparently the buttons need to be declared as global to avoid 
    # that they are garbage-collected. If this happens they will still be visible but become inactive

    def stopAnim(d):
        global stop
        stop=True

    def startAnim(d):
        global stop
        stop=False
        foo()

    def advance(d):
        global mat, config, plt, time
        time += 1
        step(threshold)
        mat.set_data(config)
        plt.title('t = ' + str(time))    
        plt.show()

    def initAnim(d):
        global mat, config, plt, time
        time = 0
        init(density)
        mat = ax.matshow(config, cmap=cm.seismic)
        mat.set_data(config)
        plt.title('t = ' + str(time))    
        plt.show()    

    def updateSpeed(val):
        global speed
        speed = 1/sspeed.val

    def foo():
        global speed, timer, stop
        advance(None)
        if not(stop):
            timer=threading.Timer(speed, foo)
            timer.start()

    
    RD.seed()
    fig, ax = plt.subplots()
    ax.axis('off')
    plt.title("Shelling's Segregation Model")

    axspeed = plt.axes([0.175, 0.05, 0.65, 0.03])
    sspeed = Slider(axspeed, 'Speed', 0.1, 10.0, valinit=1.0)
    sspeed.on_changed(updateSpeed)

    axnext = plt.axes([0.85, 0.15, 0.1, 0.075])
    axstart = plt.axes([0.85, 0.25, 0.1, 0.075])
    axstop = plt.axes([0.85, 0.35, 0.1, 0.075])
    axinit = plt.axes([0.85, 0.45, 0.1, 0.075])
    bnext = Button(axnext, 'Next')
    bnext.on_clicked(advance)
    bstart = Button(axstart, 'Start')
    bstart.on_clicked(startAnim)
    bstop = Button(axstop, 'Stop')
    bstop.on_clicked(stopAnim)
    binit = Button(axinit, 'Init')
    binit.on_clicked(initAnim)
    config = SP.zeros([height, width])
    initAnim(None)
    updateSpeed(None)

    #ani = animation.FuncAnimation(fig, advance, frames=99, interval=60, save_count=50, repeat=False)
    #plt.show()

### Define the initialisation of the state

In [8]:
def init(density):
    global config, empty, agents, unhappiness, avg_similarity, speed
    speed = 1.0
    empty =[]
    agents=[]
    unhappiness=[]
    avg_similarity=[]
    config = SP.zeros([height, width])
    for x in range(width):
        for y in range(height):
            if RD.random() < density:
                agents.append((y, x))
                if RD.random() < 0.5:
                    config[y, x] = 1
                else:
                    config[y, x] = -1
            else:
                config[y,x] = 0
                empty.append((y, x))


### Define the state transition rules

In [10]:
def step(threshold):
    global config, agents, empty, unhappiness, avg_similarity

    unhappy = 0
    similarity = 0.0

    height, width = config.shape

    sequence = list(range(len(agents)))
    RD.shuffle(sequence)
    for i in sequence:
        ... # update goes here!
    unhappiness = unhappiness+[?????] # append current value for later plotting
    avg_similarity = avg_similarity+[????] # append current value for later plotting

# Auxiliary Functions for Plotting Results

In [12]:
import plotly
from plotly.offline import download_plotlyjs, init_notebook_mode, iplot
import plotly.graph_objs as go

import numpy as np
import scipy
from scipy import stats 
from IPython.display import clear_output

In [17]:
plotly.offline.init_notebook_mode(connected=True)

In [18]:
def plot_schelling():
    global unhappiness, avg_similarity
    n = len(unhappiness)
    trace1 = go.Scatter(
        x=list(range(n)), 
        y=unhappiness, 
        mode = 'lines+markers',
        name='average unhappiness'
    )
    trace2 = go.Scatter(
        x=list(range(n)), 
        y=avg_similarity, 
        mode = 'lines+markers',
        name='average similarity'
    )
    data = [trace1, trace2]
    return iplot(data)


### Set the simulation parameters

In [20]:
width = 50
height = 50

density =0.8
threshold =0.7

In [22]:
unhappiness=[]
avg_similarity=[]

# Testing

**comment these lines out if the file is imported into another file**

In [29]:
simulation(density,threshold)

/var/folders/x6/z_zgzfbs415f2sq_t2300lkc0000gr/T/ipykernel_47503/686966750.py:65: DeprecationWarning:

scipy.zeros is deprecated and will be removed in SciPy 2.0.0, use numpy.zeros instead

/var/folders/x6/z_zgzfbs415f2sq_t2300lkc0000gr/T/ipykernel_47503/3655273801.py:8: DeprecationWarning:

scipy.zeros is deprecated and will be removed in SciPy 2.0.0, use numpy.zeros instead

/var/folders/x6/z_zgzfbs415f2sq_t2300lkc0000gr/T/ipykernel_47503/686966750.py:21: UserWarning:

Starting a Matplotlib GUI outside of the main thread will likely fail.



**If you are using the TkAgg frontend, you will need to close the simulation window before the plot generated with the next call can and will be displayed**

In [30]:
#plot_schelling()